# 005 — Error Analysis on Labeled Soundscapes

Analyze exp 012 model predictions on 66 labeled soundscape files.

**Prereq**: Run inference first:
```bash
python scripts/infer_soundscapes.py --model all --split labeled
```

In [ ]:
import csv
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

DATA_DIR = Path("/storage/project/ps-clef2026_birdclef-0/shared/data/birdclef-2026")
INFERENCE_DIR = Path("/storage/project/ps-clef2026_birdclef-0/shared/inference/birdclef-2026")
WEIGHTS_DIR = Path(
    "/storage/project/ps-clef2026_birdclef-0/mgustineli3/birdclef-2026"
    "/user/murilo/experiments/010-submission-pipeline/artifacts/weights"
)

MODELS = {
    "ProtoCLR": "protoclr-2026-soundscape",
    "ConvNeXt": "convnext-2026-soundscape",
    "EAT": "eat-2026-soundscape",
}

## Load ground truth + taxonomy

In [ ]:
taxonomy = pd.read_csv(DATA_DIR / "taxonomy.csv")
all_species = sorted(taxonomy["primary_label"].astype(str).tolist())
sp2idx = {sp: i for i, sp in enumerate(all_species)}
sp2taxon = dict(zip(taxonomy["primary_label"].astype(str), taxonomy["class_name"], strict=False))
print(f"Taxonomy: {len(all_species)} species")
print(f"  {taxonomy['class_name'].value_counts().to_dict()}")

# Parse labels -> (filename, end_sec) -> set of species
segments = {}
with open(DATA_DIR / "train_soundscapes_labels.csv") as f:
    for row in csv.DictReader(f):
        fn = row["filename"]
        parts = row["end"].split(":")
        end_sec = int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])
        key = (fn, end_sec)
        if key not in segments:
            segments[key] = set()
        for sp in str(row["primary_label"]).split(";"):
            segments[key].add(sp.strip())

labeled_species = sorted({sp for spp in segments.values() for sp in spp} & set(all_species))
print(f"Segments: {len(segments)}, species covered: {len(labeled_species)}/{len(all_species)}")

## Load predictions + build aligned matrices

In [ ]:
# Load first model to get segment keys
first_slug = list(MODELS.values())[0]
ref = np.load(INFERENCE_DIR / first_slug / "labeled.npz", allow_pickle=True)
seg_keys_arr = ref["seg_keys"]  # (n, 2) object array: (filename, end_sec)
species_arr = ref["species"]  # (234,) sorted species codes

# Build GT matrix aligned to inference segment order
y_true = np.zeros((len(seg_keys_arr), len(all_species)), dtype=np.float32)
for i, (fn, end_sec) in enumerate(seg_keys_arr):
    key = (fn, int(end_sec))
    if key in segments:
        for sp in segments[key]:
            if sp in sp2idx:
                y_true[i, sp2idx[sp]] = 1.0

# Load predictions
y_pred = {}
for name, slug in MODELS.items():
    data = np.load(INFERENCE_DIR / slug / "labeled.npz", allow_pickle=True)
    y_pred[name] = data["predictions"]
    print(f"{name}: {y_pred[name].shape}")

y_pred["Ensemble"] = np.mean([y_pred[n] for n in MODELS], axis=0)
print(f"\nSegments: {len(seg_keys_arr)}, GT mean labels/seg: {y_true.sum(1).mean():.1f}")

## Per-species AUROC

In [ ]:
results = []
for sp_idx, sp in enumerate(all_species):
    col = y_true[:, sp_idx]
    n_pos = int(col.sum())
    if n_pos == 0 or n_pos == len(col):
        continue
    row = {"species": sp, "n_pos": n_pos, "taxon": sp2taxon.get(sp, "Unknown")}
    for m in list(MODELS.keys()) + ["Ensemble"]:
        row[f"auroc_{m}"] = roc_auc_score(col, y_pred[m][:, sp_idx])
    results.append(row)

results_df = pd.DataFrame(results).sort_values("auroc_Ensemble", ascending=True)
print(f"Species with AUROC computable: {len(results_df)}/{len(labeled_species)}")
print(f"\nMacro AUROC ({len(results_df)} species):")
for m in list(MODELS.keys()) + ["Ensemble"]:
    print(f"  {m}: {results_df[f'auroc_{m}'].mean():.4f}")

## Per-taxon breakdown

In [ ]:
for taxon in sorted(results_df["taxon"].unique()):
    sub = results_df[results_df["taxon"] == taxon]
    print(f"\n{taxon} ({len(sub)} species):")
    for m in list(MODELS.keys()) + ["Ensemble"]:
        col = f"auroc_{m}"
        print(
            f"  {m}: {sub[col].mean():.4f}  "
            f"(min={sub[col].min():.3f}, max={sub[col].max():.3f})"
        )

## Worst and best species

In [ ]:
display_cols = [
    "species",
    "taxon",
    "n_pos",
    "auroc_Ensemble",
    "auroc_ProtoCLR",
    "auroc_ConvNeXt",
    "auroc_EAT",
]

print("=== 15 WORST species (Ensemble AUROC) ===")
print(results_df.head(15)[display_cols].to_string(index=False))

print("\n=== 15 BEST species (Ensemble AUROC) ===")
print(results_df.tail(15)[display_cols].to_string(index=False))

## Species coverage gap

In [ ]:
uncovered = sorted(set(all_species) - set(labeled_species))
uncovered_taxa = pd.Series([sp2taxon.get(sp, "Unknown") for sp in uncovered]).value_counts()
print(f"Species NOT in labeled soundscapes: {len(uncovered)}/234")
print(f"  {uncovered_taxa.to_dict()}")

labeled_taxa = pd.Series([sp2taxon.get(sp, "Unknown") for sp in labeled_species]).value_counts()
print(f"\nSpecies IN labeled soundscapes: {len(labeled_species)}/234")
print(f"  {labeled_taxa.to_dict()}")

## Confidence: positive vs negative segments

In [ ]:
sp_idxs = [sp2idx[sp] for sp in labeled_species]
for m in list(MODELS.keys()) + ["Ensemble"]:
    p = y_pred[m][:, sp_idxs]
    gt = y_true[:, sp_idxs]
    pos_vals = p[gt == 1]
    neg_vals = p[gt == 0]
    print(
        f"{m}: pos mean={pos_vals.mean():.4f} median={np.median(pos_vals):.4f}  "
        f"neg mean={neg_vals.mean():.4f}  "
        f"sep={pos_vals.mean() / (neg_vals.mean() + 1e-8):.0f}x"
    )

## Mel frequency coverage

In [ ]:
print("Model frequency coverage:")
for name, slug in MODELS.items():
    with open(WEIGHTS_DIR / slug / "mel_config.json") as f:
        cfg = json.load(f)
    fmax = cfg.get("fmax") or cfg["sr"] // 2
    print(f"  {name}: sr={cfg['sr']}Hz, fmax={fmax}Hz")

## Ensemble vs best single model

In [ ]:
results_df["best_single"] = results_df[[f"auroc_{n}" for n in MODELS]].max(axis=1)
results_df["ensemble_gain"] = results_df["auroc_Ensemble"] - results_df["best_single"]

helped = (results_df["ensemble_gain"] > 0.01).sum()
hurt = (results_df["ensemble_gain"] < -0.01).sum()
print(
    f"Ensemble helps (>0.01): {helped}, "
    f"hurts (<-0.01): {hurt}, "
    f"neutral: {len(results_df) - helped - hurt}"
)
print(f"Mean ensemble gain: {results_df['ensemble_gain'].mean():.4f}")

## Summary

In [ ]:
n_files = len({k[0] for k in seg_keys_arr})
print("=" * 60)
print("ERROR ANALYSIS SUMMARY")
print("=" * 60)
print(f"Segments: {len(seg_keys_arr)}, Files: {n_files}")
print(f"Species evaluable: {len(results_df)}/{len(all_species)}")
print()
print("Macro AUROC:")
for m in list(MODELS.keys()) + ["Ensemble"]:
    print(f"  {m}: {results_df[f'auroc_{m}'].mean():.4f}")
print()
print("Per-taxon (Ensemble):")
for taxon in sorted(results_df["taxon"].unique()):
    sub = results_df[results_df["taxon"] == taxon]
    print(f"  {taxon} ({len(sub)} sp): {sub['auroc_Ensemble'].mean():.4f}")
print()
print(f"Not evaluable: {len(uncovered)} species ({uncovered_taxa.to_dict()})")